# Lokalisierung

06 liefert je Anfrage eine Rangliste von Datenbankbildern. Die eigentliche
Frage ist aber nicht "ist ein richtiger dabei", sondern "wo wurde das Foto
aufgenommen" -- und darauf ist die Antwort eine Koordinate.

Dieses Notebook vergleicht drei Wege von der Rangliste zur Position und
misst, wie viele Meter daneben sie liegen.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

plt.rcParams["figure.dpi"] = 150


PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config
from src.geo import to_metric_xy
from src.run_guard import (
    embedding_fingerprint,
    print_run_header,
    require_fingerprint,
    validate_config,
)

CFG = load_config(PROJECT_ROOT)
validate_config(CFG)

METHOD = CFG["vpr"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
EMBEDDING_NAME = embedding_name(CFG)

LOC = CFG.get("localization", {})
TOP_K = int(LOC.get("top_k", 10))
EPS_M = float(LOC.get("eps_m", CFG["vpr"]["uncertain_radius_m"]))
MIN_SAMPLES = int(LOC.get("min_samples", 2))
GATE_SHARE = float(LOC.get("gate_share", 0.7))
# Standard: nur Top-1. Die Aggregationsverfahren sind gemessen und
# unterliegen alle -- wer das nachpruefen will, schaltet sie ein.
COMPARE_AGGREGATIONS = bool(LOC.get("compare_aggregations", False))
TEMPERATUR = float(LOC.get("softmax_temperature", 50.0))
UNCERTAIN_RADIUS_M = float(CFG["vpr"]["uncertain_radius_m"])

RESULT_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures" / "localization"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"
retrieval_path = RESULT_DIR / "retrieval" / METHOD / f"{EMBEDDING_NAME}_retrieval.npz"

embedding_metadata = pd.read_parquet(metadata_path)
FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)
require_fingerprint(retrieval_path, FINGERPRINT, what="Retrieval-Ergebnis")

retrieval = np.load(retrieval_path)
retrieved_indices = retrieval["indices"][:, :TOP_K]
similarities = retrieval["similarities"][:, :TOP_K]

if retrieval["indices"].shape[1] < TOP_K:
    raise ValueError(
        f"localization.top_k={TOP_K}, gespeichert sind aber nur "
        f"{retrieval['indices'].shape[1]} Treffer je Anfrage."
    )

database_metadata = embedding_metadata[
    (embedding_metadata["split"] == "database").to_numpy()
].reset_index(drop=True)
query_metadata = embedding_metadata[
    (embedding_metadata["split"] == "query").to_numpy()
].reset_index(drop=True)

print_run_header(CFG, "08_localization")
print(f"Anfragen:        {len(query_metadata):,}")
print(f"Treffer je Anfrage: {TOP_K}")
print(f"DBSCAN:          eps={EPS_M:g} m, min_samples={MIN_SAMPLES}")

## Koordinaten in Meter

DBSCAN und Mittelwerte brauchen ein metrisches System -- in Grad waere ein
Radius von 25 m je nach Breitengrad unterschiedlich gross. Datenbank und
Anfragen werden gemeinsam projiziert, damit beide dieselbe Referenz haben.

In [ ]:
db_xy, UTM_CRS = to_metric_xy(
    database_metadata["lat"].to_numpy(), database_metadata["lon"].to_numpy()
)
query_xy, _ = to_metric_xy(
    query_metadata["lat"].to_numpy(), query_metadata["lon"].to_numpy(), crs=UTM_CRS
)
print(f"Projiziert nach {UTM_CRS.name}")


## Fuenf Wege von der Rangliste zur Koordinate

**Top-1** nimmt die Position des besten Treffers. Einfach, wirft aber die
uebrigen neun weg -- ein einziger falscher Erstplatzierter setzt die
Schaetzung komplett daneben.

**Schwerpunkt** mittelt alle Treffer, gewichtet nach Aehnlichkeit. Nutzt
mehr Information und glaettet das GPS-Rauschen der Referenzbilder. Die
bekannte Falle: Zerfallen die Treffer in zwei Gruppen -- sieben am Neumarkt,
drei am Hauptbahnhof -- landet der Schwerpunkt dazwischen, an einem Ort, den
kein einziger Treffer stuetzt.

**Clustering** soll genau das abfangen: die Treffer raeumlich gruppieren,
die staerkste Gruppe waehlen und erst innerhalb dieser mitteln.

Ob mehr Information tatsaechlich hilft, haengt davon ab, wie oft die Treffer
ueberhaupt richtig sind. Die Tabelle weiter unten entscheidet das -- nicht
die Plausibilitaet der Idee.

**Snap** nimmt den besten echten Treffer aus der staerksten Gruppe statt
ihres Mittelpunkts. **Gated** behaelt Top-1 und weicht nur auf die Gruppe
aus, wenn sie sich zu `localization.gate_share` einig ist. Beide sind
naheliegende Hybride -- und beide unterliegen Top-1, siehe Tabelle unten.


In [ ]:
def gewichte(sim, art="softmax"):
    """
    Cosinus-Werte liegen dicht beieinander. Ohne Spreizung waere die
    Gewichtung praktisch ein ungewichteter Mittelwert -- gemessen lagen die
    Gewichte ueber die Top-10 zwischen 0.0994 und 0.1011.
    """
    if art == "roh":
        return np.clip(sim, 0, None)
    z = TEMPERATUR * (sim - sim.max())
    return np.exp(z)


def aggregiere(punkte, sim, methode, art_gewichte="softmax"):
    """punkte: (k, 2) in Metern, sim: (k,) -> geschaetzte Position (2,)"""
    if methode == "top1":
        return punkte[0]

    if methode == "schwerpunkt":
        w = gewichte(sim, art_gewichte)
        return (punkte * w[:, None]).sum(0) / w.sum()

    if methode == "cluster":
        labels = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(punkte)
        gueltig = labels >= 0
        if not gueltig.any():
            # Keine Gruppe gefunden: die Treffer liegen alle einzeln, dann
            # ist der beste Treffer die ehrlichste Antwort.
            return punkte[0]
        # Nach summiertem Gewicht, nicht nach roher Summe: sonst gewinnt
        # die groessere Gruppe allein durch ihre Anzahl, auch wenn der beste
        # Treffer in einer kleineren sitzt. Gemessen: Median 752 m gegen
        # 662 m auf EigenPlaces.
        w_alle = gewichte(sim, art_gewichte)
        beste = max(set(labels[gueltig]), key=lambda k: w_alle[labels == k].sum())
        m = labels == beste
        w = gewichte(sim[m], art_gewichte)
        return (punkte[m] * w[:, None]).sum(0) / w.sum()

    # Zwei naheliegende Hybride, beide gemessen und beide schlechter als
    # Top-1 -- sie stehen hier, damit das Ergebnis reproduzierbar ist.
    #
    # snap:  statt zu mitteln den besten ECHTEN Treffer aus der staerksten
    #        Gruppe nehmen. Liegt Top-1 nicht darin, ist meist die Gruppe
    #        falsch, nicht Top-1 -- deshalb genauso schlecht wie Mitteln.
    # gated: Top-1 behalten, die Gruppe nur nehmen, wenn sie sich zu
    #        GATE_SHARE einig ist. Naehert sich Top-1 von unten, nie darueber.
    if methode in ("snap", "gated"):
        labels = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(punkte)
        gueltig = labels >= 0
        if not gueltig.any():
            return punkte[0]
        w_alle = gewichte(sim, art_gewichte)
        beste = max(set(labels[gueltig]), key=lambda k: w_alle[labels == k].sum())
        m = labels == beste
        if methode == "snap":
            return punkte[np.flatnonzero(m)[0]]
        if m.mean() < GATE_SHARE:
            return punkte[0]
        w = gewichte(sim[m], art_gewichte)
        return (punkte[m] * w[:, None]).sum(0) / w.sum()

    raise ValueError(f"Unbekannte Methode: {methode}")


def cluster_anteil(punkte, sim):
    """Anteil der Treffer in der staerksten Gruppe -- ein Konfidenzmass."""
    labels = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(punkte)
    gueltig = labels >= 0
    if not gueltig.any():
        return 0.0
    # Dieselbe Wahl wie in aggregiere: nach gewichteter Summe, sonst
    # misst die Konfidenz eine andere Gruppe als die Schaetzung nutzt.
    w = gewichte(sim)
    beste = max(set(labels[gueltig]), key=lambda k: w[labels == k].sum())
    return float((labels == beste).mean())

## Fehler je Verfahren

Fuer jede Anfrage der Abstand zwischen geschaetzter und echter Position.
Berichtet wird der Median statt des Mittelwerts, weil eine Handvoll voellig
danebenliegender Schaetzungen den Mittelwert sonst dominiert.

In [ ]:
from tqdm.auto import tqdm

VARIANTEN = [("Top-1", "top1", "softmax")]
if COMPARE_AGGREGATIONS:
    VARIANTEN += [
        ("Schwerpunkt (roh)", "schwerpunkt", "roh"),
        ("Schwerpunkt (gespreizt)", "schwerpunkt", "softmax"),
        ("Clustering", "cluster", "softmax"),
        ("Snap (bester Treffer der Gruppe)", "snap", "softmax"),
        ("Gated (Gruppe nur bei Einigkeit)", "gated", "softmax"),
    ]

fehler = {name: np.empty(len(query_metadata)) for name, _, _ in VARIANTEN}
anteil_dominant = np.empty(len(query_metadata))

for qi in tqdm(range(len(query_metadata)), desc="Lokalisieren"):
    punkte = db_xy[retrieved_indices[qi]]
    sim = similarities[qi]
    wahr = query_xy[qi]

    for name, methode, art in VARIANTEN:
        geschaetzt = aggregiere(punkte, sim, methode, art)
        fehler[name][qi] = np.linalg.norm(geschaetzt - wahr)

    if COMPARE_AGGREGATIONS:
        anteil_dominant[qi] = cluster_anteil(punkte, sim)

print("fertig")

In [ ]:
# Zwei triviale Vergleichswerte, damit die Meter einzuordnen sind.
rng = np.random.default_rng(CFG["vpr"]["split_seed"])
zufall = db_xy[rng.integers(0, len(db_xy), len(query_xy))]
fehler["Zufaelliges DB-Bild"] = np.linalg.norm(zufall - query_xy, axis=1)

mitte = db_xy.mean(axis=0)
fehler["Stadtmittelpunkt"] = np.linalg.norm(mitte - query_xy, axis=1)


def kennzahlen(e):
    return {
        "median_m": float(np.median(e)),
        "p25_m": float(np.percentile(e, 25)),
        "p75_m": float(np.percentile(e, 75)),
        "p90_m": float(np.percentile(e, 90)),
        **{f"unter_{s}m": float((e <= s).mean()) for s in (10, 25, 50, 100)},
    }


tabelle = {name: kennzahlen(e) for name, e in fehler.items()}

kopf = f"{'Verfahren':<26}{'Median':>9}{'p25':>8}{'p75':>9}{'p90':>10}" + \
       "".join(f"{'<' + str(s) + 'm':>8}" for s in (10, 25, 50, 100))
print(kopf)
print("-" * len(kopf))
for name, k in tabelle.items():
    print(f"{name:<26}{k['median_m']:>8.0f}m{k['p25_m']:>7.0f}m{k['p75_m']:>8.0f}m"
          f"{k['p90_m']:>9.0f}m" +
          "".join(f"{k[f'unter_{s}m']:>8.3f}" for s in (10, 25, 50, 100)))

print()
# ------------------------------------------------------------------
# Struktur der Fehlgriffe: WIE weit liegt ein falscher Top-1-Treffer
# daneben? Beinahetreffer (dieselbe Strasse, ein Stueck versetzt) und grobe
# Verwechslungen (anderer Stadtteil) sind verschiedene Fehler mit
# verschiedenen Gegenmitteln -- ein Umsortieren der Trefferliste kann nur
# die zweite Sorte beheben.
# ------------------------------------------------------------------
falsch = fehler["Top-1"] > UNCERTAIN_RADIUS_M
d_falsch = fehler["Top-1"][falsch]
fehlstruktur = {
    "n_falsch": int(falsch.sum()),
    "anteil_falsch": float(falsch.mean()),
    "median_m": float(np.median(d_falsch)) if falsch.any() else None,
    **{f"unter_{s}m": float((d_falsch <= s).mean()) if falsch.any() else None
       for s in (50, 100, 250, 500, 1000, 2000)},
}
print()
print(f"Falscher Top-1-Treffer (> {UNCERTAIN_RADIUS_M:g} m): "
      f"{fehlstruktur['n_falsch']:,} von {len(fehler['Top-1']):,} Anfragen "
      f"({fehlstruktur['anteil_falsch']:.1%})")
print(f"  Median des Fehlers:  {fehlstruktur['median_m']:,.0f} m")
for s in (50, 100, 500, 1000):
    print(f"  unter {s:>5} m:        {fehlstruktur[f'unter_{s}m']:.1%}")
print("  Liegt der Median im Kilometerbereich, sind die Fehler grobe")
print("  Verwechslungen, keine Beinahetreffer.")

print("Zum Einordnen: liegt Top-1 vorn, sind die Nachbartreffer zu oft falsch,")
print("als dass Mitteln helfen koennte. Der Vorsprung des Schwerpunkts mit")
print("gespreizten Gewichten gegenueber den rohen zeigt, wieviel die")
print("Gewichtung ueberhaupt ausmacht.")

## Verteilung der Fehler

Die Kurve zeigt, welcher Anteil der Anfragen unterhalb eines gegebenen
Fehlers liegt. Der Abstand zu den beiden Vergleichswerten macht sichtbar,
wieviel das Verfahren ueberhaupt leistet.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
schritte = np.logspace(0, 4.5, 200)

for name in fehler:
    e = fehler[name]
    anteil = [(e <= s).mean() for s in schritte]
    stil = "--" if name in ("Zufaelliges DB-Bild", "Stadtmittelpunkt") else "-"
    ax.plot(schritte, anteil, stil, label=name, linewidth=1.6)

for s in (10, 25, 50, 100):
    ax.axvline(s, color="0.85", linewidth=0.8, zorder=0)

ax.set_xscale("log")
ax.set_xlabel("Lokalisierungsfehler [m]")
ax.set_ylabel("Anteil der Anfragen")
ax.set_title(f"{EMBEDDING_NAME}  |  {len(query_metadata):,} Anfragen")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURE_DIR / f"{EMBEDDING_NAME}_lokalisierungsfehler.png", bbox_inches="tight")
print(f"gespeichert: {FIGURE_DIR / (EMBEDDING_NAME + '_lokalisierungsfehler.png')}")
plt.show()

## Der Zwei-Gruppen-Fall

Die Abbildung, die erklaert, warum Clustering noetig ist: eine Anfrage, bei
der der Schwerpunkt deutlich schlechter liegt als das Clustering.

In [ ]:
if not COMPARE_AGGREGATIONS:
    print("Zwei-Gruppen-Abbildung entfaellt: localization.compare_aggregations ist aus.")
else:
    unterschied = fehler["Schwerpunkt (gespreizt)"] - fehler["Clustering"]
    kandidaten = np.argsort(unterschied)[::-1]
    qi = int(kandidaten[0])

    punkte = db_xy[retrieved_indices[qi]]
    sim = similarities[qi]
    wahr = query_xy[qi]
    labels = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(punkte)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    for gruppe in sorted(set(labels)):
        m = labels == gruppe
        ax.scatter(*punkte[m].T, s=70,
                   label=f"Gruppe {gruppe}" if gruppe >= 0 else "ohne Gruppe",
                   alpha=0.85, edgecolor="white", zorder=3)

    ax.scatter(*wahr, marker="*", s=420, color="#1565c0",
               label="echte Position", zorder=5, edgecolor="white")
    ax.scatter(*aggregiere(punkte, sim, "schwerpunkt"), marker="X", s=200,
               color="#c62828", label="Schwerpunkt", zorder=5, edgecolor="white")
    ax.scatter(*aggregiere(punkte, sim, "cluster"), marker="P", s=200,
               color="#2e7d32", label="Clustering", zorder=5, edgecolor="white")

    ax.set_aspect("equal")
    ax.set_xlabel("Ost [m]")
    ax.set_ylabel("Nord [m]")
    ax.set_title(
        f"Query {qi}: Schwerpunkt {fehler['Schwerpunkt (gespreizt)'][qi]:,.0f} m, "
        f"Clustering {fehler['Clustering'][qi]:,.0f} m"
    )
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / f"{EMBEDDING_NAME}_zwei_gruppen.png", bbox_inches="tight")
    plt.show()

## Konfidenz aus der Geschlossenheit

Der Anteil der Treffer in der staerksten Gruppe ist ein Mass dafuer, wie
einig sich das Retrieval ist. Liegt er hoch, ist die Schaetzung meist gut --
das ist bereits das dritte Signal fuer die spaetere OOD-Ablehnung.

In [ ]:
if not COMPARE_AGGREGATIONS:
    print("Konfidenz-Abbildung entfaellt: localization.compare_aggregations ist aus.")
else:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    stufen = [(0.0, 0.4), (0.4, 0.7), (0.7, 1.01)]
    for unten, oben in stufen:
        m = (anteil_dominant >= unten) & (anteil_dominant < oben)
        if m.sum() < 50:
            continue
        ax.hist(np.clip(fehler["Clustering"][m], 1, 1e4), bins=np.logspace(0, 4, 40),
                histtype="step", linewidth=1.8, density=True,
                label=f"{unten:.0%}-{min(oben,1):.0%} in der Gruppe  (n={m.sum():,}, "
                      f"Median {np.median(fehler['Clustering'][m]):,.0f} m)")

    ax.set_xscale("log")
    ax.set_xlabel("Lokalisierungsfehler [m]")
    ax.set_ylabel("Dichte")
    ax.set_title("Geschlossenheit der Treffer gegen tatsaechlichen Fehler")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / f"{EMBEDDING_NAME}_konfidenz.png", bbox_inches="tight")
    plt.show()

In [ ]:
EVAL_PATH = RESULT_DIR / "localization" / f"{EMBEDDING_NAME}.json"
EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_PATH.write_text(
    json.dumps(
        {
            "datum": pd.Timestamp.now().strftime("%Y-%m-%d"),
            "method": METHOD,
            "adapter": ADAPTER,
            "embedding_name": EMBEDDING_NAME,
            "top_k": TOP_K,
            "eps_m": EPS_M,
            "min_samples": MIN_SAMPLES,
            "n_queries": int(len(query_metadata)),
            "verfahren": tabelle,
            "fehlstruktur_top1": fehlstruktur,
        },
        indent=2,
    )
)
print(f"gespeichert: {EVAL_PATH}")